# MedMNIST v2 — Phase 2 Reproduction
## Person A: ResNet-18 on DermaMNIST (28×28, --resize to 224×224)

**Target metrics (from paper Table 1):**
- AUC: ~0.917
- Accuracy: ~0.754

**Requirements:** Enable GPU (P100/T4) and Internet in Kaggle notebook settings before running.

### Cell 1 — GPU Check

In [ ]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM           : {mem_gb:.1f} GB")
else:
    raise RuntimeError("No GPU found — enable GPU in Kaggle notebook settings (Settings → Accelerator → GPU P100)")

### Cell 2 — Install Missing Packages

In [ ]:
# Kaggle pre-installs: torch, torchvision, numpy, pandas, scikit-learn, Pillow
# We only need to add these three:
!pip install -q medmnist tensorboardX fire
print("Packages installed.")

### Cell 3 — Clone Experiments Repo

In [ ]:
import os

os.chdir('/kaggle/working')
!git clone https://github.com/MedMNIST/experiments
os.chdir('/kaggle/working/experiments/MedMNIST2D')
print("Working directory:", os.getcwd())
print("Files:", os.listdir('.'))

### Cell 4 — Training (expect 1–2 hrs on P100)

In [ ]:
import time

os.makedirs('/kaggle/working/output', exist_ok=True)
torch.cuda.reset_peak_memory_stats()

t_start = time.time()

!python train_and_eval_pytorch.py \
    --data_flag dermamnist \
    --output_root /kaggle/working/output \
    --num_epochs 100 \
    --gpu_ids 0 \
    --batch_size 128 \
    --resize \
    --model_flag resnet18 \
    --download \
    --run run1

t_total = time.time() - t_start
print(f"\nTotal training time : {t_total/3600:.2f} h  ({t_total/60:.0f} min)")
print(f"Approx time/epoch   : {t_total/100/60:.1f} min")

### Cell 5 — Read Aggregate Results from Log

In [ ]:
import glob

log_files = glob.glob('/kaggle/working/output/dermamnist/**/*.txt', recursive=True)
print(f"Found {len(log_files)} log file(s)\n")
for f in log_files:
    print(f"=== {f} ===")
    with open(f) as fp:
        print(fp.read())

print("\n--- Paper targets ---")
print("  AUC (test) : ~0.917")
print("  Acc (test) : ~0.754")

### Cell 6 — GPU Memory Usage

In [ ]:
if torch.cuda.is_available():
    peak_alloc   = torch.cuda.max_memory_allocated(0)  / 1e9
    peak_reserved = torch.cuda.max_memory_reserved(0) / 1e9
    print(f"Peak GPU memory allocated : {peak_alloc:.2f} GB")
    print(f"Peak GPU memory reserved  : {peak_reserved:.2f} GB")
    print(f"GPU                       : {torch.cuda.get_device_name(0)}")

### Cell 7 — Per-Class Analysis (Extension Hook for Phase 3)
The paper only reports aggregate AUC/Acc. This cell adds per-class AUC, accuracy, and F1.

In [ ]:
import sys
import numpy as np
import PIL
import pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet18
from medmnist import DermaMNIST
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize

CLASS_NAMES = [
    'Actinic keratoses',
    'Basal cell carcinoma',
    'Benign keratosis',
    'Dermatofibroma',
    'Melanoma',
    'Melanocytic nevi',
    'Vascular lesions',
]
N_CLASSES = 7
DEVICE    = torch.device('cuda:0')

# Load best model
model_files = glob.glob('/kaggle/working/output/dermamnist/**/best_model.pth', recursive=True)
assert model_files, "best_model.pth not found — check that training completed"
model_path = sorted(model_files)[-1]  # latest run if multiple
print(f"Loading: {model_path}")

model = resnet18(weights=None, num_classes=N_CLASSES).to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE)['net'])
model.eval()

# Transform matches --resize training flag (28→224)
transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=PIL.Image.NEAREST),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5]),
])

test_ds     = DermaMNIST(split='test', transform=transform, download=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

all_scores, all_targets = [], []
with torch.no_grad():
    for inputs, targets in test_loader:
        scores = torch.softmax(model(inputs.to(DEVICE)), dim=1)
        all_scores.append(scores.cpu().numpy())
        all_targets.append(targets.numpy().flatten())

y_score = np.vstack(all_scores)    # (N, 7)  softmax probabilities
y_true  = np.concatenate(all_targets)  # (N,)
y_pred  = y_score.argmax(axis=1)   # (N,)

# Per-class AUC (one-vs-rest)
y_bin = label_binarize(y_true, classes=list(range(N_CLASSES)))
per_auc = [roc_auc_score(y_bin[:, i], y_score[:, i]) for i in range(N_CLASSES)]

# Per-class accuracy from confusion matrix diagonal
cm = confusion_matrix(y_true, y_pred)
per_acc = cm.diagonal() / cm.sum(axis=1)

# Per-class F1, Precision, Recall
per_f1  = f1_score(y_true, y_pred, average=None, zero_division=0)
per_prec = precision_score(y_true, y_pred, average=None, zero_division=0)
per_rec  = recall_score(y_true, y_pred, average=None, zero_division=0)
counts   = np.bincount(y_true, minlength=N_CLASSES)

# Build results table
df = pd.DataFrame({
    'Class':     CLASS_NAMES,
    'N (test)':  counts,
    'AUC':       np.round(per_auc,  4),
    'Accuracy':  np.round(per_acc,  4),
    'Precision': np.round(per_prec, 4),
    'Recall':    np.round(per_rec,  4),
    'F1':        np.round(per_f1,   4),
})

print("\n=== Per-Class Results — ResNet-18, DermaMNIST (Test Set) ===")
print(df.to_string(index=False))
print("\n--- Aggregate ---")
print(f"  Macro AUC : {np.mean(per_auc):.4f}   (paper: ~0.917)")
print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}   (paper: ~0.754)")
print(f"  Macro F1  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")

# Save for Phase 3
df.to_csv('/kaggle/working/perclass_resnet18_dermamnist.csv', index=False)
print("\nSaved: /kaggle/working/perclass_resnet18_dermamnist.csv")

### Cell 8 — Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

short = ['Actinic\nkerat.', 'Basal cell\ncarc.', 'Benign\nkerat.',
         'Dermato-\nfibroma', 'Melanoma', 'Melanocytic\nnevi', 'Vascular\nlesions']

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(short, fontsize=7.5)
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(short, fontsize=7.5)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('ResNet-18 — DermaMNIST Confusion Matrix (Test Set)')

thresh = cm.max() / 2
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8,
                color='white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_resnet18_dermamnist.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrix_resnet18_dermamnist.png")

### Cell 9 — Per-Class AUC Bar Chart

In [ ]:
colors = ['#e07b39','#4878d0','#6acc65','#d65f5f','#b47cc7','#c4ad66','#77bedb']
short_names = ['Actinic\nkerat.', 'Basal cell\ncarc.', 'Benign\nkerat.',
               'Dermato-\nfibroma', 'Melanoma', 'Melanocytic\nnevi', 'Vascular\nlesions']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# AUC
bars = axes[0].bar(range(N_CLASSES), per_auc, color=colors, edgecolor='white')
axes[0].axhline(np.mean(per_auc), color='black', linestyle='--', linewidth=1,
                label=f'Macro AUC = {np.mean(per_auc):.3f}')
axes[0].axhline(0.917, color='red', linestyle=':', linewidth=1,
                label='Paper target (0.917)')
axes[0].set_xticks(range(N_CLASSES))
axes[0].set_xticklabels(short_names, fontsize=8)
axes[0].set_ylim(0.5, 1.0)
axes[0].set_ylabel('AUC'); axes[0].set_title('Per-Class AUC (One-vs-Rest)')
axes[0].legend(fontsize=8)
for b, v in zip(bars, per_auc):
    axes[0].text(b.get_x() + b.get_width()/2, v + 0.005, f'{v:.3f}',
                 ha='center', fontsize=7)

# Accuracy
bars2 = axes[1].bar(range(N_CLASSES), per_acc, color=colors, edgecolor='white')
axes[1].axhline(accuracy_score(y_true, y_pred), color='black', linestyle='--',
                linewidth=1, label=f'Overall Acc = {accuracy_score(y_true, y_pred):.3f}')
axes[1].axhline(0.754, color='red', linestyle=':', linewidth=1,
                label='Paper target (0.754)')
axes[1].set_xticks(range(N_CLASSES))
axes[1].set_xticklabels(short_names, fontsize=8)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Accuracy'); axes[1].set_title('Per-Class Accuracy')
axes[1].legend(fontsize=8)
for b, v in zip(bars2, per_acc):
    axes[1].text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}',
                 ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('/kaggle/working/perclass_metrics_resnet18_dermamnist.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: perclass_metrics_resnet18_dermamnist.png")

### Cell 10 — Summary Table (Our Results vs. Paper)

In [ ]:
our_auc = float(np.mean(per_auc))
our_acc = float(accuracy_score(y_true, y_pred))

summary = pd.DataFrame({
    'Config':   ['ResNet-18, DermaMNIST, --resize'],
    'AUC (paper)':  [0.917],
    'AUC (ours)':   [round(our_auc, 4)],
    'AUC diff':     [round(our_auc - 0.917, 4)],
    'Acc (paper)':  [0.754],
    'Acc (ours)':   [round(our_acc, 4)],
    'Acc diff':     [round(our_acc - 0.754, 4)],
})

print("\n=== Reproduction Summary ===")
print(summary.to_string(index=False))

summary.to_csv('/kaggle/working/reproduction_summary_personA.csv', index=False)
print("\nSaved: reproduction_summary_personA.csv")
print("\n--- Files to download from Kaggle output ---")
for f in glob.glob('/kaggle/working/*.csv') + glob.glob('/kaggle/working/*.png'):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):<55} {size_kb:>7.1f} KB")